# 02 — Neural ODEs

**Mathematical Foundations of Control and Machine Learning — Winter 2026**

In this notebook, we explore the connection between neural networks and differential equations. As we push the depth to a continuum limit and obtain the **neural ODE** (Chen et al., 2018): the neural network learns a vector field that defines the dynamics of the system. This places deep learning squarely inside the world of dynamical systems and control.

1. **From ResNets to ODEs** — depth as continuous time
2. **Solving ODEs numerically** — Euler and Runge–Kutta methods
3. **Differentiating through the solver** — learning a vector field from data
4. **Neural ODEs** — an MLP as the right-hand side
5. **The adjoint method** — gradients from a backward ODE
6. **Continuous-depth classification** — deforming the plane with a flow


In [54]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

# Reproducibility
rng = np.random.default_rng(0)
torch.manual_seed(0)

# A small, colorblind-safe palette used throughout the notebook
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
plt.rcParams.update({
    "figure.figsize": (6, 4), "figure.dpi": 100,
    "axes.prop_cycle": plt.cycler(color=[BLUE, ORANGE, AQUA, YELLOW, MAGENTA]),
    "axes.grid": False, "grid.color": "#e1e0d9", "axes.edgecolor": "#c3c2b7",
    "axes.spines.top": False, "axes.spines.right": False,
})

print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.0+cpu


## 1. From residual networks to ODEs

A **residual network** (ResNet) does not learn a map from scratch at every layer — it learns the *increment*. Layer $k$ updates its hidden state $x_k \in \mathbb{R}^d$ by

$$x_{k+1} = x_k + f_{\theta_k}(x_k).$$

Compare this with the **explicit Euler method** for the ODE $\dot x = f_\theta(x)$ with step size $h$:

$$x_{k+1} = x_k + h\, f_\theta(x_k).$$

A ResNet *is* an Euler discretization of a differential equation. Letting the number of layers grow and $h \to 0$, the network becomes the **flow** of an ODE. This is the **neural ODE ansatz**:

$$\begin{cases}\dot x(t) = f_\theta\big(x(t)\big), \\ x(0) = x_{\text{in}}.\end{cases}$$

Instead of designing an architecture layer by layer, we choose a parametrized **vector field** $f_\theta : \mathbb{R}^d \to \mathbb{R}^d$ and let it act for time $T$. 

Training is an optimal control problem. The parameters $\theta$ *steer* the trajectories so that a loss function of the endpoint is minimized.

We will use neural ODEs for: 
1. **learning dynamics**, when the data *is* a trajectory and we want the law of motion behind it,
2. **learning representations**, when the flow deforms the input space until a simple linear ansatz can finish the job.

## 2. Solving ODEs numerically

An **initial value problem** asks for a function $x(t)$ with

$$\dot x(t) = f\big(t, x(t)\big), \qquad x(t_0) = x_0.$$

If $f$ is Lipschitz in $x$, a unique solution exists (Picard–Lindelöf). 

No closed-form solutions available, so we approximate it on a time grid $t_0 < t_1 < \dots < t_N$ with steps $h = t_{k+1} - t_k$.

- **Explicit Euler** follows the tangent: $x_{k+1} = x_k + h\, f(t_k, x_k)$. Its local error per step is $\mathcal{O}(h^2)$, the accumulated **global error is $\mathcal{O}(h)$** — *first order*.

- **Classical Runge–Kutta (RK4)** samples the slope four times per step and combines the samples:
$$
k_1 = f(t_k, x_k), \quad
k_2 = f\Big(t_k + \tfrac h2,\, x_k + \tfrac h2 k_1\Big), \quad
k_3 = f\Big(t_k + \tfrac h2,\, x_k + \tfrac h2 k_2\Big), \quad
k_4 = f(t_k + h,\, x_k + h k_3),
$$
$$x_{k+1} = x_k + \frac h6\big(k_1 + 2k_2 + 2k_3 + k_4\big),$$
with **global error $\mathcal{O}(h^4)$** — *fourth order*: halving $h$ buys a factor of 16 in accuracy.

In [55]:
def euler_step(f, t, x, h):
    return x + h * f(t, x)

def rk4_step(f, t, x, h):
    k1 = f(t, x)
    k2 = f(t + h / 2, x + h / 2 * k1)
    k3 = f(t + h / 2, x + h / 2 * k2)
    k4 = f(t + h, x + h * k3)
    return x + h / 6 * (k1 + 2 * k2 + 2 * k3 + k4)

def odeint(f, x0, ts, step=rk4_step):
    """Integrate dx/dt = f(t, x) through the time grid ts.
    """
    xs = [x0]
    for t0, t1 in zip(ts[:-1], ts[1:]):
        xs.append(step(f, t0, xs[-1], t1 - t0))
    return torch.stack(xs)

Let us consider the **Lorenz system**

$$\begin{cases}\dot x_1 = \sigma (x_2 - x_1), \\ \dot x_2 = x_1 (\rho - x_3) - x_2, \\ \dot x_3 = x_1 x_2 - \beta x_3,\end{cases}$$

with the classical parameters $\sigma = 10$, $\rho = 28$, $\beta = 8/3$.

In [ ]:
# Test problem: the Lorenz system, a chaotic 3-D flow
SIGMA, RHO, BETA = 10.0, 28.0, 8.0 / 3.0

def f_lorenz(t, x):
    x1, x2, x3 = x[..., 0], x[..., 1], x[..., 2]
    return torch.stack([SIGMA * (x2 - x1), x1 * (RHO - x3) - x2, x1 * x2 - BETA * x3], dim=-1)

x0 = torch.tensor([1.0, 1.0, 1.0])
ts = torch.linspace(0, 3, 201)                  # h = 0.02
with torch.no_grad():
    tr_euler = odeint(f_lorenz, x0, ts, euler_step)
    tr_rk4 = odeint(f_lorenz, x0, ts, rk4_step)

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(tr_euler[:, 0], tr_euler[:, 2], color=ORANGE, lw=1.5, label="Euler")
ax.plot(tr_rk4[:, 0], tr_rk4[:, 2], color=BLUE, lw=1.5, label="RK4")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_3$")
ax.legend(fontsize=8, loc="upper left"); ax.set_title("Lorenz system, $(x_1, x_3)$ projection")
plt.show()

## 4. neural ODE: Learning dynamics

Real dynamics are rarely linear. The neural ODE replaces the linear ansatz with a small MLP,

$$\dot x = f_\theta(x), \qquad f_\theta : \mathbb{R}^3 \to \mathbb{R}^3 \ \text{a neural network},$$

which by universal approximation can represent essentially any smooth vector field. 

In [ ]:
# Vector field ansatz: a small MLP with tanh activations.
class VectorField(nn.Module):
    """The neural ODE ansatz: an MLP playing the role of the right-hand side."""
    def __init__(self, dim=2, width=64):
        super().__init__()           # typical size of the state, keeps tanh out of saturation
        self.net = nn.Sequential(
            nn.Linear(dim, width), nn.Tanh(),
            nn.Linear(width, width), nn.Tanh(),
            nn.Linear(width, dim),
        )
    def forward(self, t, x):          # same signature as every f used by odeint
        return self.net(x)

For the dataset, we generate a training trajectory with RK4 and add noise:

In [ ]:
ts_lor = torch.linspace(0, 20, 2001)                # h = 0.01
with torch.no_grad():
    x_lor = odeint(f_lorenz, torch.tensor([1.0, 1.0, 1.0]), ts_lor)
    x_lor = x_lor + 0.1 * torch.randn_like(x_lor)

Neural networks train best when their inputs and outputs are of order one, but Lorenz states are of order $10$ and velocities of order $100$. Rather than rescaling inside the network, we rescale the **dynamical system**. With a normalized state and a new clock,

$$z = D^{-1}(x - \mu), \qquad \tau = s\, t, \qquad D = \operatorname{diag}(\text{std of the data}),$$

the chain rule gives

$$\frac{dz}{d\tau} = \frac{1}{s}\, D^{-1} f_{\text{Lorenz}}(\mu + D z) =: g(z).$$

This system is **conjugate** to Lorenz: the same trajectories, in new coordinates and on a new clock. We fit a plain `VectorField` to $g$ on the normalized data and map predictions back with $x = \mu + D z$.

In [ ]:
# Rescaled system: normalized state z = (x - mu) / sd, new time tau = TAU_SCALE * t
TAU_SCALE = 10.0
mu, sd = x_lor.mean(0), x_lor.std(0)
z_lor, taus = (x_lor - mu) / sd, TAU_SCALE * ts_lor

f_theta = VectorField(dim=3, width=128)
optimizer = torch.optim.Adam(f_theta.parameters(), lr=3e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=2000)

SEG, BATCH = 10, 64                   # window of 10 steps = 0.1 time units
node_losses = []
for it in range(2001):
    i0 = torch.randint(0, len(taus) - SEG, (BATCH,))
    target = z_lor[i0[:, None] + torch.arange(SEG + 1)]
    pred = odeint(f_theta, z_lor[i0], taus[: SEG + 1])
    L = torch.mean((pred.transpose(0, 1) - target) ** 2)        # loss function 
    optimizer.zero_grad()
    L.backward()
    optimizer.step()
    scheduler.step()
    node_losses.append(L.item())

In [ ]:
x0_new = torch.tensor([-11.0, -19.0, 17.0])         # near the attractor, but not a training point
ts_new = torch.linspace(0, 3, 501)
with torch.no_grad():
    tr_true = odeint(f_lorenz, x0_new, ts_new)      # ground truth: RK4 on the true Lorenz field
    tr_hat = mu + sd * odeint(f_theta, (x0_new - mu) / sd, TAU_SCALE * ts_new)  # learned g, mapped back

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.plot(x_lor[:, 0], x_lor[:, 2], ".", color="#b8b7ad", ms=1, label="training data")
ax.plot(tr_true[:, 0], tr_true[:, 2], color=BLUE, lw=1.5, label="RK4, true Lorenz field")
ax.plot(tr_hat[:, 0], tr_hat[:, 2], color=ORANGE, lw=1.5, label=r"RK4, learned $f_\theta$")
ax.plot(x0_new[0], x0_new[2], "o", color="#0b0b0b", ms=6, label="new initial condition")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_3$")
ax.legend(fontsize=8, loc="upper left", ncol=2)
ax.set_title("Neural ODE test")
plt.show()

## 6. Neural ODEs: learning representations

Our goal is to classify data using a neural ODE:

$$\text{logits}(x_{\text{in}}) = W\, x(1) + b, \qquad \dot x = f_\theta(x), \quad x(0) = x_{\text{in}},$$

trained with the cross-entropy loss function. An affine layer can only draw a **straight** decision boundary, so all the geometric work must be done by the flow: it has to *deform the plane* until the two moons are linearly separable.

In [ ]:
def make_moons(n_per_class=200, noise=0.08):        # as in notebook 01
    t = np.pi * rng.random(n_per_class)
    upper = np.stack([np.cos(t), np.sin(t)], axis=1)             # class 0
    lower = np.stack([1 - np.cos(t), 0.5 - np.sin(t)], axis=1)   # class 1
    X = np.concatenate([upper, lower]) + noise * rng.standard_normal((2 * n_per_class, 2))
    y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)]).astype(np.int64)
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y)

X_moons, y_moons = make_moons()
perm = torch.randperm(len(X_moons))
tr_idx, te_idx = perm[:300], perm[300:]              # 75% train / 25% test

field = VectorField(width=32)                        # the flow's right-hand side
head = nn.Linear(2, 2)                               # affine readout on x(1)
ts_flow = torch.linspace(0, 1, 11)                   # T = 1, ten RK4 steps

def logits(x):
    return head(odeint(field, x, ts_flow)[-1])

optimizer = torch.optim.Adam(list(field.parameters()) + list(head.parameters()), lr=1e-2)
loss_fn = nn.CrossEntropyLoss()

for it in range(501):
    L = loss_fn(logits(X_moons[tr_idx]), y_moons[tr_idx])
    optimizer.zero_grad()
    L.backward()
    optimizer.step()
    if it % 100 == 0:
        with torch.no_grad():
            acc = (logits(X_moons[te_idx]).argmax(dim=1) == y_moons[te_idx]).float().mean()
        print(f"iteration {it:3d}:  loss = {L.item():.4f},  test accuracy = {acc:.1%}")

iteration   0:  loss = 0.8869,  test accuracy = 33.0%
iteration 100:  loss = 0.0138,  test accuracy = 99.0%
iteration 200:  loss = 0.0001,  test accuracy = 100.0%
iteration 300:  loss = 0.0000,  test accuracy = 100.0%
iteration 400:  loss = 0.0000,  test accuracy = 100.0%
iteration 500:  loss = 0.0000,  test accuracy = 100.0%


### Animation: riding the flow

Every input point follows its own solution of $\dot x = f_\theta(x)$, and the flow deforms the plane until the classes are linearly separable. The dashed line is the straight boundary $w^\top x + b = 0$ of the affine readout at $t = 1$; pulled back through the flow, it becomes the curved boundary in input space shown below.

In [ ]:
# The trajectory of EVERY data point under the learned flow
with torch.no_grad():
    flow_traj = odeint(field, X_moons, torch.linspace(0, 1, 41)).numpy()   # (41, 400, 2)
    w = (head.weight[0] - head.weight[1]).numpy()                          # readout boundary w·x + b = 0
    b = (head.bias[0] - head.bias[1]).item()
colors = np.where(y_moons.numpy() == 0, BLUE, ORANGE)

fig, ax = plt.subplots(figsize=(5.5, 4))
lo, hi = flow_traj.min(axis=(0, 1)) - 0.3, flow_traj.max(axis=(0, 1)) + 0.3
ax.set(xlim=(lo[0], hi[0]), ylim=(lo[1], hi[1]), xlabel="$x_1$", ylabel="$x_2$")
x1 = np.array([lo[0], hi[0]])
ax.plot(x1, -(w[0] * x1 + b) / w[1], "--", color="#0b0b0b", lw=1.5)
sc = ax.scatter(*flow_traj[0].T, c=colors, s=10)

def update(k):
    sc.set_offsets(flow_traj[k])
    ax.set_title(f"flow at $t = {k / 40:.2f}$")

anim = animation.FuncAnimation(fig, update, frames=len(flow_traj), interval=120)
plt.close(fig)
HTML(anim.to_jshtml())

In [ ]:
# Decision boundary back in INPUT space: grid points are flowed forward, then classified
g1, g2 = np.meshgrid(np.linspace(-1.8, 2.8, 120), np.linspace(-1.3, 1.8, 120))
grid = torch.tensor(np.stack([g1.ravel(), g2.ravel()], axis=1), dtype=torch.float32)
with torch.no_grad():
    probs = torch.softmax(logits(grid), dim=1)[:, 1].reshape(g1.shape).numpy()

plt.contourf(g1, g2, probs, levels=20, cmap="RdBu_r", alpha=0.55, vmin=0, vmax=1)
plt.colorbar(label=r"$P(\mathrm{class}\ 1 \mid x)$")
plt.contour(g1, g2, probs, levels=[0.5], colors="#0b0b0b", linewidths=1.5)
plt.scatter(*X_moons.T, c=colors, s=14, edgecolors="white", lw=0.4)
plt.xlabel("$x_1$"); plt.ylabel("$x_2$")
plt.title("Decision boundary of the neural ODE classifier")
plt.show()